# Plot scGPT

In [1]:
%load_ext autoreload
%autoreload 2
import os
import pickle
import polars as pl
import pandas as pd
from deeptan.stat.metrics import ClusteringMetricsCalculator, transform_ct_df, read_batch_from_h5ad

/home/wuch/miniforge3/envs/sc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_true_labels(true_data_dir, seed, split: int = 2):
    _path = os.path.join(true_data_dir, f"split_{seed}_{split}.parquet")
    _obs_names = pl.read_parquet(_path).select(["obs_names"])

    _labels_df = pl.read_parquet(os.path.join(true_data_dir, "celltypes_onehot.parquet")).rename({"bc": "obs_names"})
    # Pick obs_names that are in the split and keep the order
    _labels_df = _obs_names.join(_labels_df, on="obs_names", how="left")
    _labels_ct = transform_ct_df(_labels_df)
    return _labels_ct, _labels_df, _obs_names


def load_cell_emb(path_results, _dataset, _type, seed, n_feat, _trn_grad: int):
    _path_embs = os.path.join(path_results, f"{_dataset}_{_type}_{_trn_grad}")
    _files = os.listdir(_path_embs)
    _suff = f"{seed}_{n_feat}.pkl"
    _path_emb = [i for i in _files if i.endswith(_suff)][0]
    _path_emb = os.path.join(_path_embs, _path_emb)
    _emb = pd.read_pickle(_path_emb)
    return _emb


def load_batch_info(true_data_dir, _obs_names: pl.DataFrame):
    path_h5ad = os.path.join(true_data_dir, "origin.h5ad")
    _batch_df = read_batch_from_h5ad(path_h5ad)
    _batch_df = _obs_names.join(_batch_df, on="obs_names", how="left")
    return _batch_df


# Main
def calc_clustering_metrics(method_name: str, _n_feat, _trn_grad: int, train_size: str, path_results: str, _dataset, _type, _seed, true_data_dir):
    _labels_ct, _labels_df, _obs_names = load_true_labels(true_data_dir, _seed)
    _batch_df = load_batch_info(true_data_dir, _obs_names)
    _emb = load_cell_emb(path_results, _dataset, _type, _seed, _n_feat, _trn_grad)

    metricscalc = ClusteringMetricsCalculator(
        true_labels=_labels_df.drop("obs_names").to_numpy(),
        pred_labels=None,
        batches=_batch_df["batch"].to_numpy(),
        X_emb=_emb.to_numpy(),
        n_neighbors=10,
    )
    metrics_clustering_dict = metricscalc.calculate_metrics()
    metrics_clustering_df = pl.DataFrame(metrics_clustering_dict)
    _more_info = pl.DataFrame({"method": method_name, "dataset": _dataset, "seed": _seed, "train_size": train_size, "n_feat": _n_feat})
    metrics_clustering_df = _more_info.hstack(metrics_clustering_df)
    return metrics_clustering_df

## Pre-defined parameters

In [3]:
train_size_snrna = ["5439", "2720", "1359", "680"]
train_size_scmul = ["4560", "2280", "1136", "572"]
trn_grads = [8, 4, 2, 1]

seeds = [42, 43, 44, 45, 46]
ns_feat = [400, 800, 1200, 1600, 2000]

## Select a dataset to eval

In [ ]:
method_name = "scGPT"

# _dataset = "snrna"
_dataset = "scmul"

# train_size = train_size_snrna
train_size = train_size_scmul

path_results = "/mnt/hdd2/homext/wuch/xn2p/run/predict/scgpt/"
# true_data_dir = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/scRNA/GSE226097_Annotated_split_strata"
true_data_dir = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/scMultiome/Ath_scMultiome_WT_split_strata"

_type = "cell_emb"
# _type = "gene_emb"

In [6]:
method_name = "scGPT"

_dataset = "snrna"
# _dataset = "scmul"

train_size = train_size_snrna
# train_size = train_size_scmul

path_results = "/mnt/hdd2/homext/wuch/xn2p/run/predict/scgpt/"
true_data_dir = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/scRNA/GSE226097_Annotated_split_strata"
# true_data_dir = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/scMultiome/Ath_scMultiome_WT_split_strata"

_type = "cell_emb"
# _type = "gene_emb"

## Run metrics

In [7]:
_df_list = []
for i, _trn_size in enumerate(train_size):
    for _seed in seeds:
        for _n_feat in ns_feat:
            _df_list.append(calc_clustering_metrics(method_name, _n_feat, trn_grads[i], _trn_size, path_results, _dataset, _type, _seed, true_data_dir))
_df = pl.concat(_df_list)

# path_save = f"metrics.clus.{method_name}.{_dataset}.{train_size}" + ".csv"
path_save = f"metrics.clus.{method_name}.{_dataset}" + ".csv"
_df.write_csv(path_save)